# Staffing Adequacy and Weekend Conversion Rate in Retail

**Goal:** Identify whether staffing levels correlate with conversion rate (CR) differences between weekdays and weekends, and quantify the revenue opportunity for underperforming stores.

**Key findings:**
1. All stores show a negative CR gap (weekend vs weekday) in the 15–17 time slot
2. In that slot, staffing coverage positively correlates with CR — especially on weekends
3. Five stores identified with the highest incremental revenue potential (~€1M/year at constant traffic)

**Data:** Hourly store-level data — foot traffic (entries), conversions, staff on shift, and sales. Jan–Sep 2025.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

pd.options.display.float_format = '{:.3f}'.format
sns.set_theme(style='whitegrid', palette='muted')

## 1. Load & clean data

In [ ]:
df = pd.read_csv('data/store_hourly_data.csv')

# types
df['detection_date'] = pd.to_datetime(df['detection_date'])
df['amount'] = df['amount'].astype('float')

# derived columns
df['is_weekend'] = df['day_of_week_num'].isin([6, 7]).astype(int)
df['cr'] = df['conversioni'] / df['ingressi']
df['staff_coverage_per_100_entries'] = (df['actual_people_on_turn'] / df['ingressi']) * 100

# cap CR at 1 and remove noise
df['cr'] = np.where(df['cr'] > 1, 1, df['cr'])
df = df[df['cr'] < 1]
df = df[(df['ora'] > 8) & (df['ora'] < 21)]
df = df[(df['amount'] > 0) & (df['actual_people_on_turn'] > 0) & (df['conversioni'] > 10)]

# time slots
conditions = [df['ora'] <= 11, df['ora'] <= 14, df['ora'] <= 17, df['ora'] > 17]
df['day_slot'] = np.select(conditions, ['morning_8-11', 'lunch_12-14', 'afternoon_15-17', 'late_afternoon_18-close'], default='other')

print(f"Rows after cleaning: {len(df):,}")
print(f"Stores: {df['store_id'].nunique()} | Date range: {df['detection_date'].min().date()} → {df['detection_date'].max().date()}")
df.head(3)

## 2. CR gap by time slot: weekend vs weekday

We compute the median CR for each store × hour combination, separately for weekdays and weekends, then compute the difference.

In [ ]:
measures = ['cr', 'staff_coverage_per_100_entries', 'ingressi', 'conversioni', 'amount']

pivot = df[['store_id', 'store_desc', 'cluster_desc', 'region_desc', 'ora', 'day_slot', 'is_weekend'] + measures].copy()

pivoted = pivot.pivot_table(
    index=['store_id', 'store_desc', 'cluster_desc', 'region_desc', 'ora', 'day_slot'],
    columns='is_weekend',
    values=measures,
    aggfunc=['median', 'mean', 'sum']
).reset_index()

# flatten columns
pivoted.columns = [
    '_'.join(filter(None, map(str, col))).strip('_')
    if isinstance(col, tuple) else str(col)
    for col in pivoted.columns
]

pivoted['median_cr_diff'] = pivoted['median_cr_1'] - pivoted['median_cr_0']
pivoted['median_staff_diff'] = pivoted['median_staff_coverage_per_100_entries_1'] - pivoted['median_staff_coverage_per_100_entries_0']

pivoted.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: median CR diff by hour, colored by slot
slot_order = ['morning_8-11', 'lunch_12-14', 'afternoon_15-17', 'late_afternoon_18-close']
slot_colors = {'morning_8-11': '#4C72B0', 'lunch_12-14': '#DD8452', 'afternoon_15-17': '#55A868', 'late_afternoon_18-close': '#C44E52'}

for slot in slot_order:
    sub = pivoted[pivoted['day_slot'] == slot].groupby('ora')['median_cr_diff'].mean().reset_index()
    axes[0].bar(sub['ora'], sub['median_cr_diff'], label=slot, color=slot_colors[slot], alpha=0.85)

axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Median CR difference (weekend vs weekday) by hour')
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Median CR diff')
axes[0].legend(fontsize=8)

# KDE: distribution of CR diff by slot
for slot in slot_order:
    sub = pivoted[pivoted['day_slot'] == slot]['median_cr_diff'].dropna()
    x = np.linspace(sub.min(), sub.max(), 200)
    axes[1].plot(x, stats.gaussian_kde(sub)(x), label=slot, color=slot_colors[slot])

axes[1].axvline(0, color='grey', linestyle='--')
axes[1].set_title('Distribution of median CR diff by time slot')
axes[1].set_xlabel('median_cr_diff')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/cr_gap_by_slot.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding 1:** The afternoon slot (15–17) is the **only** slot where 100% of store–hour combinations show a negative CR gap on weekends (mean gap: −6.3pp). Other slots are essentially noise around zero — morning, lunch and late afternoon all sit around 57–60% negative, consistent with random variation rather than a structural pattern.

## 3. Staffing coverage vs CR: correlation by slot and weekend flag

We plot staffing coverage (staff per 100 entries) against CR, split by time slot and weekday/weekend.

In [ ]:
plot_df = df[
    (df['staff_coverage_per_100_entries'] < 30) &
    (df['ora'] < 20)
].copy()
plot_df['is_weekend_label'] = plot_df['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

slot_order = ['morning_8-11', 'lunch_12-14', 'afternoon_15-17', 'late_afternoon_18-close']
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True, sharex=True)

for row_idx, is_we in enumerate([0, 1]):
    for col_idx, slot in enumerate(slot_order):
        ax = axes[row_idx][col_idx]
        sub = plot_df[(plot_df['is_weekend'] == is_we) & (plot_df['day_slot'] == slot)]
        ax.scatter(sub['staff_coverage_per_100_entries'], sub['cr'], alpha=0.3, s=10, color='#C44E52' if is_we else '#4C72B0')
        if len(sub) > 10:
            m, b, r, p, _ = stats.linregress(sub['staff_coverage_per_100_entries'], sub['cr'])
            x_line = np.linspace(sub['staff_coverage_per_100_entries'].min(), sub['staff_coverage_per_100_entries'].max(), 100)
            ax.plot(x_line, m * x_line + b, color='black', linewidth=1.2)
            ax.set_title(f"{slot}\n{'Weekend' if is_we else 'Weekday'} | r={r:.2f}, p={p:.3f}", fontsize=8)
        ax.set_xlabel('Staff / 100 entries', fontsize=7)
        ax.set_ylabel('CR', fontsize=7)

plt.suptitle('CR vs Staffing Coverage by Time Slot and Day Type', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('outputs/cr_vs_staffing_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding 2:** The staffing → CR correlation is **r = 0.49 on weekend afternoons**, vs r ≈ 0.07–0.09 in every other slot and day combination. The effect is specific: on weekends in the 15–17 slot, stores are on average understaffed by −5.8 staff/100 entries relative to their weekday levels, and CR drops accordingly. Every other moment of the week shows no meaningful relationship between staffing coverage and CR.

## 4. Store-level ranking: anomaly score and incremental opportunity

We weight the CR gap by traffic volume to get a measure of how much revenue is being left on the table in each slot.

In [ ]:
tgt = pivoted[['store_id', 'store_desc', 'cluster_desc', 'region_desc', 'ora', 'day_slot',
               'median_cr_0', 'median_cr_1', 'median_cr_diff',
               'median_staff_coverage_per_100_entries_0', 'median_staff_coverage_per_100_entries_1',
               'median_staff_diff',
               'median_ingressi_0', 'median_ingressi_1']].copy()

# z-score of CR diff
mean_val = tgt['median_cr_diff'].mean()
std_val = tgt['median_cr_diff'].std(ddof=0)
tgt['zscore_cr_diff'] = (tgt['median_cr_diff'] - mean_val) / std_val

# traffic weight within store (how busy is this slot relative to the store's peak)
tgt['traffic_weight'] = tgt['median_ingressi_1'] / tgt.groupby('store_id')['median_ingressi_1'].transform('max')
tgt['anomaly_score'] = tgt['zscore_cr_diff'] * tgt['traffic_weight']

# ranking by slot
ranking = (
    tgt.groupby(['store_id', 'store_desc', 'cluster_desc', 'region_desc', 'day_slot'])[
        ['anomaly_score', 'median_cr_diff', 'median_cr_0', 'median_cr_1', 'median_ingressi_0', 'median_ingressi_1']
    ].mean()
    .reset_index()
)

ranking['avg_incremental_conversions_per_hour'] = np.where(
    ranking['median_cr_diff'] < 0,
    abs(ranking['median_cr_diff']) * ranking['median_ingressi_1'],
    0
)

ranking.sort_values('avg_incremental_conversions_per_hour', ascending=False)

## 5. Store selection: incremental opportunity vs CR baseline

Two criteria drive store selection:
1. **Incremental conversion potential** — how many extra conversions per hour if the weekend CR gap were closed
2. **Weekday CR baseline** — avoid stores that are already over-performing; focus on those with room to grow

The scatter below shows all stores on both dimensions. Stores in the top-left quadrant have high opportunity and an aggredible baseline.

In [ ]:
# global CR by store (weekday baseline)
global_cr = (
    df.groupby(['store_id', 'store_desc', 'is_weekend'])[['ingressi', 'conversioni']]
    .sum()
    .reset_index()
)
global_cr['cr_global'] = global_cr['conversioni'] / global_cr['ingressi']
weekday_baseline = (
    global_cr[global_cr['is_weekend'] == 0][['store_id', 'cr_global']]
    .rename(columns={'cr_global': 'cr_baseline_weekday'})
)

# pivot ranking by slot
slot_cols_rename = {
    'morning_8-11': 'extra_conv_morning_8-11',
    'lunch_12-14': 'extra_conv_lunch_12-14',
    'afternoon_15-17': 'extra_conv_afternoon_15-17',
    'late_afternoon_18-close': 'extra_conv_late_18-close'
}

final = ranking.pivot_table(
    index=['region_desc', 'store_id', 'cluster_desc', 'store_desc'],
    columns='day_slot',
    values='avg_incremental_conversions_per_hour',
    aggfunc='sum'
).reset_index()

slot_cols = [c for c in ['morning_8-11', 'lunch_12-14', 'afternoon_15-17', 'late_afternoon_18-close'] if c in final.columns]
final['total_incremental_conversions_per_hour'] = final[slot_cols].sum(axis=1)
final = final.rename(columns=slot_cols_rename)

final = pd.merge(final, weekday_baseline, on='store_id', how='left')
final = final.sort_values('total_incremental_conversions_per_hour', ascending=False)

print(f"All {len(final)} stores:")
final[['region_desc', 'store_id', 'cluster_desc', 'store_desc',
       'extra_conv_afternoon_15-17', 'total_incremental_conversions_per_hour', 'cr_baseline_weekday']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Scatter: all stores, incremental opportunity vs CR baseline ---
ax = axes[0]
ax.scatter(
    final['cr_baseline_weekday'],
    final['total_incremental_conversions_per_hour'],
    s=60, color='#4C72B0', alpha=0.8, zorder=3
)

# label every store
for _, row in final.iterrows():
    ax.annotate(
        row['store_desc'],
        xy=(row['cr_baseline_weekday'], row['total_incremental_conversions_per_hour']),
        xytext=(4, 2), textcoords='offset points',
        fontsize=6.5, alpha=0.85
    )

# reference lines: median of each axis
med_cr = final['cr_baseline_weekday'].median()
med_inc = final['total_incremental_conversions_per_hour'].median()
ax.axvline(med_cr, color='grey', linestyle='--', linewidth=0.8, label=f'Median CR baseline ({med_cr:.2f})')
ax.axhline(med_inc, color='grey', linestyle=':', linewidth=0.8, label=f'Median opportunity ({med_inc:.1f})')

ax.set_xlabel('Weekday CR baseline')
ax.set_ylabel('Total incremental conversions / hour (if CR gap = 0)')
ax.set_title('Store selection: opportunity vs CR baseline\n(top-left = high gap, aggredible baseline)')
ax.legend(fontsize=8)

# --- Bar: top 10 by total incremental ---
ax2 = axes[1]
top10 = final.head(10)
bars = ax2.barh(top10['store_desc'], top10['total_incremental_conversions_per_hour'], color='#55A868')
ax2.set_xlabel('Avg incremental conversions / hour')
ax2.set_title('Top 10 stores by total incremental opportunity')
ax2.invert_yaxis()
for bar, val in zip(bars, top10['total_incremental_conversions_per_hour']):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/store_opportunity_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


**Store selection logic:** stores are prioritized by combining high incremental conversion potential *and* a weekday CR baseline that is not already saturated. A store with a weekday CR above ~0.73 is deprioritized: even if it shows a weekend gap, the absolute room to grow is limited and the cost of misallocating staff is higher. The actionable targets sit in the top-left quadrant of the scatter — high gap, mid baseline.

## 6. Linear regression: does more staffing actually move CR?

Simple OLS regression to confirm the direction and magnitude of the staffing → CR effect.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

reg_df = df[(df['conversioni'] > 10) & (df['actual_people_on_turn'] > 0)].copy()

numeric_features = ['staff_coverage_per_100_entries', 'ingressi']
categorical_features = ['day_slot']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
], remainder='drop').set_output(transform='pandas')

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
]).set_output(transform='pandas')

X = reg_df[numeric_features + categorical_features]
y = reg_df['conversioni']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(f"R² train: {pipe.score(X_train, y_train):.3f}")
print(f"R² test:  {r2_score(y_test, y_pred):.3f}")
print(f"MAE:      {mean_absolute_error(y_test, y_pred):.2f} conversions")

feat_names = pipe.named_steps['preprocessor'].get_feature_names_out()
coefs = pipe.named_steps['regressor'].coef_
coef_df = pd.DataFrame({'feature': feat_names, 'coefficient': coefs}).sort_values('coefficient', ascending=False)
print("\nCoefficients:")
print(coef_df)

**Finding 3:** The OLS regression (R² = 0.71 on both train and test sets) confirms the direction: `staff_coverage_per_100_entries` carries a coefficient of **+0.51**, meaning each additional unit of staffing coverage is associated with ~0.5 extra conversions per hour, after controlling for traffic volume and time slot. The afternoon_15-17 slot dummy has a negative intercept (−0.67) relative to other slots, consistent with the structural CR gap observed in the exploratory phase.

---

## Next step

One store implemented a shift redistribution on weekends (no new headcount). The causal impact is measured using a Difference-in-Differences approach in the companion repo: [`did-staffing-intervention`](https://github.com/marcomorabito94-mm/staffing-conversion-rate-analysis).